# Linear Programming with Multiplicative Weights: Fractional Flow Routing
In this activity, we use the multiplicative weights update algorithm to route a fixed amount of flow across a capacitated network, and then to estimate the largest amount of flow the network can carry.

> __Learning Objectives.__
>
> By the end of this activity, you will be able to:
>
> * __Model routing as a feasibility problem:__ Split a fixed throughput across competing routes so that no link exceeds its capacity, expressed as a simplex feasibility problem.
> * __Estimate maximum throughput by binary search:__ Repeatedly call the feasibility solver while raising the target throughput to approximate the largest routable flow.
> * __Compare the approximate result to an exact solver:__ Check the multiplicative weights estimate against the exact maximum flow from a linear programming solver.

Let's get started!
___

## Background: routing on the simplex
We route a total throughput $\tau$ from a source to a sink by splitting it across $m$ candidate routes. Let $x_{i}\ge 0$ be the flow assigned to route $i$, with $\sum_{i} x_{i} = \tau$. Each network link $k$ has a capacity $b_{k}$, and the entry $a_{ki}$ equals one when route $i$ uses link $k$.

> A routing is admissible when no link is overloaded: we want $\mathbf{x}\in\Delta_{m} = \{\mathbf{x}\ge 0 : \sum_{i} x_{i} = \tau\}$ with $\mathbf{A}\mathbf{x}\le\mathbf{b}$, where $a_{ki}\in\{0,1\}$ records which links each route uses and $b_{k}$ is the capacity of link $k$. This is the same simplex feasibility problem solved by the multiplicative weights algorithm.

We reuse [the `MyLinearProgramFeasibilityProblem` type](src/Types.jl) and its [`solve(...)` method](src/Solve.jl) from the module's `src/` folder. Let's describe the network.
___

<div>
    <center>
        <img src="figs/flow-routing-tikz/flow-routing.svg" width="780"/>
    </center>
</div>

## The network
Consider a network carrying flow from a source $s$ to a sink $t$ over four routes. The routes share some links:

> __Links and capacities.__
>
> * __Trunk (e0, capacity 12):__ used by all four routes.
> * __Sub-trunk (eL, capacity 5):__ shared by routes 1 and 2.
> * __Branches (e1–e4, capacities 2, 6, 6, 6):__ one private link per route; route 1's branch is the tightest at capacity 2.

The link–route incidence matrix $\mathbf{A}\in\mathbb{R}^{6\times 4}$ has one row per link and one column per route, with $a_{ki}=1$ when route $i$ uses link $k$. The capacity vector is $\mathbf{b} = (12, 5, 2, 6, 6, 6)$. Let's load the environment and build the problem.
___

## Setup, Data, and Prerequisites
We include the `Include.jl` file to load the required packages and the module's local `src/` codes.

In [1]:
include("Include.jl"); # load my codes, packages, etc

### Route a target throughput
Let's first ask whether the network can carry a target throughput of $\tau = 10$. We build [a `MyLinearProgramFeasibilityProblem`](src/Types.jl) with the incidence matrix, the capacities, and $\tau$, then solve it. We store the problem in `routing_problem`.

In [2]:
routing_problem = let

    # link–route incidence A[k,i] = 1 if route i uses link k -
    A = [
        1.0  1.0  1.0  1.0 ;  # e0 trunk     (all routes)
        1.0  1.0  0.0  0.0 ;  # eL sub-trunk (routes 1,2)
        1.0  0.0  0.0  0.0 ;  # e1 branch    (route 1)
        0.0  1.0  0.0  0.0 ;  # e2 branch    (route 2)
        0.0  0.0  1.0  0.0 ;  # e3 branch    (route 3)
        0.0  0.0  0.0  1.0 ;  # e4 branch    (route 4)
    ];
    b = [12.0, 5.0, 2.0, 6.0, 6.0, 6.0]; # link capacities
    τ = 10.0;                            # target throughput

    problem = build(MyLinearProgramFeasibilityProblem, (A = A, b = b, τ = τ, ϵ = 0.01));
    problem; # return the problem
end;

In [3]:
routing_result = solve(routing_problem);

Can the network carry $\tau = 10$ units, and how is the flow split across the routes?

In [4]:
let
    x = routing_result["x"];
    println("feasible = ", routing_result["feasible"], "   (iterations = ", routing_result["iterations"], ")");

    df = DataFrame();
    for i ∈ eachindex(x)
        push!(df, (route = "P$(i)", flow = x[i]));
    end
    push!(df, (route = "total", flow = sum(x)));
    pretty_table(df;
        backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact));
end

feasible = true   (iterations = 2)
 -------- ---------
   route      flow 
  String   Float64 
 -------- ---------
      P1   1.42857
      P2   2.85714
      P3   2.85714
      P4   2.85714
   total      10.0
 -------- ---------


> __Try it.__ Re-run the build cell with a larger target, for example $\tau = 12$ or $\tau = 13$. The value $\tau = 12$ is still routable, but $\tau = 13$ is not: the solver runs to its iteration limit and reports `feasible = false`. The trunk capacity of 12 caps the total flow.

Rather than change the target by hand, let's sweep a range of throughputs and record where feasibility ends.

### Where is the feasibility boundary?
We sweep the target throughput and record whether each value is routable. We expect feasibility up to the trunk capacity, then infeasibility beyond it.

In [5]:
let
    A = routing_problem.A;
    b = routing_problem.b;

    df = DataFrame();
    for τ in [8.0, 10.0, 11.0, 12.0, 12.5, 13.0]
        r = solve(build(MyLinearProgramFeasibilityProblem, (A = A, b = b, τ = τ, ϵ = 0.01)));
        push!(df, (target_τ = τ, feasible = r["feasible"], iterations = r["iterations"]));
    end
    pretty_table(df;
        backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact));
end

 ---------- ---------- ------------
  target_τ   feasible   iterations 
   Float64       Bool        Int64 
 ---------- ---------- ------------
       8.0       true            1
      10.0       true            2
      11.0       true            2
      12.0       true            2
      12.5      false        13863
      13.0      false        13863
 ---------- ---------- ------------


## Maximum throughput by binary search
The feasibility solver answers _"is throughput $\tau$ routable?"_ We can turn this into an optimization: find the largest routable $\tau$. Because scaling a feasible routing down by a factor in $(0,1]$ keeps it feasible, feasibility is monotone in $\tau$, so we can binary search. We keep the largest $\tau$ the solver certifies as feasible.

In [6]:
max_throughput = let
    A = routing_problem.A;
    b = routing_problem.b;

    τ_lo = 0.0;          # known routable
    τ_hi = sum(b);       # loose upper bracket
    x_best = zeros(size(A, 2));
    for step in 1:60
        τ_mid = 0.5*(τ_lo + τ_hi);
        r = solve(build(MyLinearProgramFeasibilityProblem, (A = A, b = b, τ = τ_mid, ϵ = 0.005)));
        if (r["feasible"] == true)
            τ_lo = τ_mid;       # routable: raise the floor
            x_best = r["x"];
        else
            τ_hi = τ_mid;       # not routable: lower the ceiling
        end
    end

    println("MWA maximum routable throughput ≈ ", round(τ_lo, digits = 3));
    println("route flows at the maximum       = ", round.(x_best, digits = 3));
    τ_lo; # return the estimate
end;

MWA maximum routable throughput ≈ 12.01
route flows at the maximum       = [1.092, 2.184, 4.367, 4.367]


### Check against an exact solver
The exact maximum flow solves the linear program $\max\,\sum_{i} x_{i}$ subject to $\mathbf{A}\mathbf{x}\le\mathbf{b}$ and $\mathbf{x}\ge 0$. We solve it with the GLPK solver through [the `MyLinearProgrammingProblemModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) exported by the package and compare.

In [7]:
glpk_max_flow = let
    A = routing_problem.A;
    b = routing_problem.b;
    m = size(A, 2);

    lp = build(MyLinearProgrammingProblemModel, (
        A = A, b = b, c = ones(m), lb = zeros(m), ub = fill(1.0e6, m)));
    result = VLDataScienceMachineLearningPackage.solve(lp; constraints = :leq);

    println("GLPK exact maximum flow = ", round(result["objective_value"], digits = 3));
    result["objective_value"]; # return the exact value
end;

GLPK exact maximum flow = 12.0


The multiplicative weights estimate should sit just below the exact maximum flow: the algorithm certifies _interior_ routings, while the exact maximum saturates a network cut, a point on the boundary of the feasible set. Let's confirm the two agree to within a couple of percent.

In [8]:
let
    gap = abs(max_throughput - glpk_max_flow) / glpk_max_flow;
    println("MWA estimate = ", round(max_throughput, digits = 3));
    println("GLPK exact   = ", round(glpk_max_flow, digits = 3));
    println("relative gap = ", round(100*gap, digits = 2), " %");

    @assert gap < 0.02 # MWA estimate within 2% of the exact maximum flow
end

MWA estimate = 12.01
GLPK exact   = 12.0
relative gap = 0.08 %


> __Try it.__ Increase the trunk capacity (the first entry of $\mathbf{b}$) and re-run the maximum-throughput and exact-solver cells. Because the branch links can together carry more than 12 units, raising the trunk lets the network move more flow, and both the multiplicative weights estimate and the exact maximum flow increase together.

___

## Summary
In this activity, we used the multiplicative weights algorithm to route a fixed throughput across a capacitated network, then estimated the maximum routable flow by binary search and checked it against an exact solver.

> __Key Takeaways:__
>
> * __Routing is a feasibility problem:__ Splitting a fixed throughput across routes without overloading any link is a simplex feasibility problem the multiplicative weights algorithm solves directly.
> * __Optimization from feasibility:__ Because feasibility is monotone in the throughput, a binary search over the target turns the feasibility solver into an estimator of the maximum flow.
> * __Approximate meets exact:__ The multiplicative weights estimate sits just below the exact maximum flow, because it certifies interior routings while the true maximum saturates a network cut.

The feasibility solver, wrapped in a binary search, recovers the network's maximum throughput to within a couple of percent of the exact value. The next activity pushes this idea further, using the same feasibility engine to maximize a profit objective.
___